# Market Research with LLM-based Synthetic Consumers
Implementation of Semantic Similarity Rating (SSR) method from [Maier et al. 2025](https://arxiv.org/html/2510.08338v3)

This notebook implements a complete pipeline for evaluating visual advertisements using synthetic consumers powered by Azure OpenAI.


## 1. Configuration - Adverts to Eval

In [ ]:
# Show the adverts side by side
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# ===== ADVERTS TO EVALUATE =====
# List of paths to visual adverts (can be URLs or local file paths)
ADVERTS = [
    "https://ui0arpl8sm.ufs.sh/f/Hzww9EQvYURJynPRFKukVH2MI5Lo4ehEfAXvZdcmtWqPg7rp",
    "https://ui0arpl8sm.ufs.sh/f/Hzww9EQvYURJ4LdzSDlYIif5CL8BKvMsOh2ZnmS7yHt0jTD3",
]

# Labels for each advert (optional, for visualization)
ADVERT_LABELS = [
    "Landing page A",
    "Landing page B",
]

# ===== INTENT QUESTION =====
INTENT_QUESTION = "signup intent"

# Full question template (will be formatted with demographics)
QUESTION_TEMPLATES = {
    "signup intent": "Based on this landing page, how likely would you be to sign up to play this online game? Please consider carefully, and remember you already pushed an advertisement for a 'browser based ninja game' in order to get to the landing page.",
    "purchase intent": "Based on this advertisement, how likely would you be to purchase this product if it were available?",
    "relevance": "How relevant is this advertisement to you and your needs?",
}

QUESTION = QUESTION_TEMPLATES.get(INTENT_QUESTION, INTENT_QUESTION)


def load_image(advert_path):
    if advert_path.startswith("http"):
        response = requests.get(advert_path)
        img = Image.open(BytesIO(response.content))
    else:
        img = Image.open(advert_path)
    return img

fig, axes = plt.subplots(1, len(ADVERTS), figsize=(len(ADVERTS) * 7, 6))
if len(ADVERTS) == 1:
    axes = [axes]
for idx, (advert_path, label) in enumerate(zip(ADVERTS, ADVERT_LABELS)):
    img = load_image(advert_path)
    axes[idx].imshow(img)
    axes[idx].axis("off")
    axes[idx].set_title(label)
plt.tight_layout()
plt.show()


## 2. Demographics & Sample Settings


In [ ]:
# ===== DEMOGRAPHICS DISTRIBUTIONS =====
# Define the demographic attributes to simulate
DEMOGRAPHICS = {
    "age_group": ["18-24", "25-34", "35-44"],
    "gender": ["male", "female", "non-binary"],
    "income_level": ["low", "middle", "upper-middle", "high"],
    "education": ["high school", "some college", "bachelor's degree", "graduate degree"],
    "region": ["urban", "suburban", "rural"],
}

# Number of synthetic consumers to generate per advert
NUM_SAMPLES = 200

# Temperature for embedding similarity computation (0.0 for deterministic)
EMBEDDING_TEMPERATURE = 0.0


## 3. Setup & Dependencies


In [ ]:
import os
import base64
import random
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import AzureOpenAI
from sklearn.metrics.pairwise import cosine_similarity

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## 4. Azure OpenAI Client Setup


In [ ]:
# Initialize Azure OpenAI clients
# Make sure to set these environment variables:
# - AZURE_OPENAI_API_KEY
# - AZURE_OPENAI_ENDPOINT
# - AZURE_OPENAI_DEPLOYMENT (e.g., "gpt-4o")
# - AZURE_OPENAI_EMBEDDING_DEPLOYMENT (e.g., "text-embedding-3-large")

os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_OPENAI_EMBEDDING_ENDPOINT"] = ""

# GPT client uses the normal endpoint
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

# Embeddings client uses the alternative endpoint
embedding_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01",
    azure_endpoint=os.getenv("AZURE_OPENAI_EMBEDDING_ENDPOINT"),
)

GPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5-2025-08-07")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")

print(f"Using GPT deployment: {GPT_DEPLOYMENT}")
print(f"Using embedding deployment: {EMBEDDING_DEPLOYMENT}")

# --- Minimal logic to verify model API works ---

def test_gpt_deployment(deployment_name: str):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Hello! Reply only with 'ok'."}
            ],
        )
        print(f"GPT deployment ({deployment_name}) Chat API test: {response.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"GPT deployment ({deployment_name}) test FAILED: {str(e)}")

def test_embedding_deployment(deployment_name: str):
    try:
        resp = embedding_client.embeddings.create(
            input="test",
            model=deployment_name
        )
        embedding = resp.data[0].embedding if resp.data else None
        if embedding:
            print(f"Embedding deployment ({deployment_name}) Embedding API test: success ({len(embedding)} dims)")
        else:
            print(f"Embedding deployment ({deployment_name}) returned no embedding")
    except Exception as e:
        print(f"Embedding deployment ({deployment_name}) test FAILED: {str(e)}")

# Run test for both deployments
test_gpt_deployment(GPT_DEPLOYMENT)
test_embedding_deployment(EMBEDDING_DEPLOYMENT)

## 5. Reference Anchor Statements
These anchor statements define the 5-point Likert scale for semantic similarity mapping (SSR method)


In [ ]:
# Reference anchor statements for 5-point Likert scale
# Based on the paper's approach: create multiple reference sets and average
REFERENCE_SETS = {
    "signup intent": [
        {
            1: "I definitely would not sign up for this service. It does not appeal to me at all.",
            2: "I probably would not sign up for this service. It has limited appeal.",
            3: "I might sign up for this service. I'm undecided and would need more information.",
            4: "I probably would sign up for this service. It seems appealing to me.",
            5: "I definitely would sign up for this service. It strongly appeals to me.",
        },
        {
            1: "This service is not for me. I would never sign up.",
            2: "I'm unlikely to sign up. This doesn't really interest me.",
            3: "I could see myself signing up, but I'm not sure yet.",
            4: "I would likely sign up. This looks interesting.",
            5: "I would absolutely sign up. This is exactly what I need.",
        },
        {
            1: "Not interested at all. I would not sign up under any circumstances.",
            2: "Somewhat uninterested. Unlikely to sign up.",
            3: "Neutral. Might consider signing up depending on other factors.",
            4: "Quite interested. Would probably sign up.",
            5: "Very interested. Would definitely sign up right away.",
        },
    ],
    "purchase intent": [
        {
            1: "I definitely would not buy this product. It does not appeal to me at all.",
            2: "I probably would not buy this product. It has limited appeal.",
            3: "I might buy this product. I'm undecided and would need more information.",
            4: "I probably would buy this product. It seems appealing to me.",
            5: "I definitely would buy this product. It strongly appeals to me.",
        },
        {
            1: "This product is not for me. I would never purchase it.",
            2: "I'm unlikely to purchase this. This doesn't really interest me.",
            3: "I could see myself buying this, but I'm not sure yet.",
            4: "I would likely purchase this. This looks interesting.",
            5: "I would absolutely buy this. This is exactly what I need.",
        },
        {
            1: "Not interested at all. I would not purchase under any circumstances.",
            2: "Somewhat uninterested. Unlikely to purchase.",
            3: "Neutral. Might consider purchasing depending on other factors.",
            4: "Quite interested. Would probably purchase.",
            5: "Very interested. Would definitely purchase right away.",
        },
    ],
    "relevance": [
        {
            1: "This is completely irrelevant to me and my needs.",
            2: "This is somewhat irrelevant to me.",
            3: "This is moderately relevant to me.",
            4: "This is quite relevant to me and my situation.",
            5: "This is extremely relevant and perfectly suits my needs.",
        },
        {
            1: "Not relevant at all. This doesn't apply to me.",
            2: "Slightly relevant, but not really for me.",
            3: "Somewhat relevant. Could be useful in some situations.",
            4: "Very relevant. This addresses my needs well.",
            5: "Extremely relevant. This is exactly what I'm looking for.",
        },
        {
            1: "No relevance whatsoever to my life or needs.",
            2: "Limited relevance to my situation.",
            3: "Moderately relevant, has some applicability.",
            4: "Highly relevant to my current needs.",
            5: "Perfect relevance. Directly addresses what I need.",
        },
    ],
}

# Get appropriate reference set based on intent question
REFERENCES = REFERENCE_SETS.get(INTENT_QUESTION, REFERENCE_SETS["signup intent"])
print(f"Using {len(REFERENCES)} reference sets for '{INTENT_QUESTION}'")


## 6. Helper Functions


In [ ]:
def generate_demographic_profile() -> Dict[str, str]:
    """Generate a random demographic profile from the defined distributions."""
    return {key: random.choice(values) for key, values in DEMOGRAPHICS.items()}


def format_demographic_prompt(demographics: Dict[str, str]) -> str:
    """Format demographics into a natural language prompt."""
    age = demographics.get("age_group", "")
    gender = demographics.get("gender", "")
    income = demographics.get("income_level", "")
    education = demographics.get("education", "")
    region = demographics.get("region", "")
    
    prompt = f"You are a {age} year old {gender} person"
    if income:
        prompt += f" with a {income} income level"
    if education:
        prompt += f" and {education} education"
    if region:
        prompt += f", living in a {region} area"
    prompt += "."
    
    return prompt


def encode_image(image_path: str) -> str:
    """Encode local image to base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def is_url(path: str) -> bool:
    """Check if path is a URL."""
    return path.startswith(("http://", "https://"))


## 7. LLM Response Generation (Textual Elicitation)


In [ ]:
def generate_textual_response(
    advert_path: str,
    demographics: Dict[str, str],
    question: str,
) -> str:
    """Generate a textual response from the LLM about an advert.
    
    This implements the textual elicitation step from the SSR method.
    """
    demographic_prompt = format_demographic_prompt(demographics)
    
    system_prompt = f"""{demographic_prompt}

You will be shown an advertisement. Please respond naturally and authentically based on your demographic profile.
Do not mention that you are an AI. Respond as if you are a real person with the described characteristics."""
    
    user_prompt = f"""{question}

Please provide your honest thoughts in 2-3 sentences, explaining your reasoning."""
    
    # Prepare image content
    if is_url(advert_path):
        image_content = {"type": "image_url", "image_url": {"url": advert_path}}
    else:
        base64_image = encode_image(advert_path)
        image_content = {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
        }
    
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                image_content,
            ],
        },
    ]
    
    response = client.chat.completions.create(
        model=GPT_DEPLOYMENT,
        messages=messages,
    )
    
    return response.choices[0].message.content.strip()


## 8. Embedding & Semantic Similarity Rating (SSR)


In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding vector for a text using Azure OpenAI."""
    response = embedding_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=text,
    )
    return np.array(response.data[0].embedding)


def compute_ssr_rating(
    response_text: str,
    reference_statements: Dict[int, str],
) -> Tuple[int, Dict[int, float]]:
    """Compute Semantic Similarity Rating (SSR) for a textual response.
    
    Args:
        response_text: The LLM's textual response
        reference_statements: Dict mapping Likert values (1-5) to reference statements
    
    Returns:
        Tuple of (predicted Likert rating, similarity scores for each reference)
    """
    # Get embedding for response
    response_embedding = get_embedding(response_text)
    
    # Get embeddings for reference statements
    similarities = {}
    for likert_value, reference_text in reference_statements.items():
        reference_embedding = get_embedding(reference_text)
        
        # Compute cosine similarity
        similarity = cosine_similarity(
            response_embedding.reshape(1, -1),
            reference_embedding.reshape(1, -1),
        )[0, 0]
        
        similarities[likert_value] = similarity
    
    # Find Likert value with highest similarity
    predicted_rating = max(similarities, key=similarities.get)
    
    return predicted_rating, similarities


def compute_mean_ssr_rating(
    response_text: str,
    reference_sets: List[Dict[int, str]],
) -> float:
    """Compute mean SSR rating across multiple reference sets.
    
    The paper recommends averaging across multiple reference sets for robustness.
    """
    ratings = []
    
    for reference_set in reference_sets:
        rating, _ = compute_ssr_rating(response_text, reference_set)
        ratings.append(rating)
    
    return np.mean(ratings)


## 9. Main Survey Execution


In [ ]:
from tqdm import tqdm
from joblib import Parallel, delayed

def sample_survey(
    advert_path: str,
    advert_label: str,
    sample_idx: int,
    question: str,
    reference_sets: List[Dict[int, str]],
):
    try:
        demographics = generate_demographic_profile()
        response_text = generate_textual_response(advert_path, demographics, question)
        ssr_rating = compute_mean_ssr_rating(response_text, reference_sets)
        result = {
            "advert": advert_path,
            "advert_label": advert_label,
            **demographics,
            "response_text": response_text,
            "ssr_rating": ssr_rating,
        }
        return result
    except Exception as e:
        tqdm.write(f"Error processing sample {sample_idx + 1} for {advert_label}: {e}")
        return None

def run_survey(
    adverts: List[str],
    advert_labels: List[str],
    num_samples: int,
    question: str,
    reference_sets: List[Dict[int, str]],
) -> pd.DataFrame:
    """Run the complete survey simulation with parallel sampling, one advert at a time.

    Returns:
        DataFrame with columns: advert, advert_label, demographics, response_text, ssr_rating
    """
    all_results = []
    for advert_idx, advert_path in enumerate(adverts):
        advert_label = advert_labels[advert_idx] if advert_idx < len(advert_labels) else f"Advert {advert_idx + 1}"
        print(f"\nEvaluating: {advert_label}")
        print(f"Path: {advert_path}")
        print("-" * 60)
        tasks = [
            (advert_path, advert_label, sample_idx, question, reference_sets)
            for sample_idx in range(num_samples)
        ]
        results = Parallel(n_jobs=8, prefer="threads")(
            delayed(sample_survey)(*task) for task in tqdm(tasks, desc=f"Samples for {advert_label}", leave=False)
        )
        results = [r for r in results if r is not None]
        all_results.extend(results)
    return pd.DataFrame(all_results)

# Run the survey
print("Starting survey simulation...")
print(f"Question: {QUESTION}")
print(f"Number of adverts: {len(ADVERTS)}")
print(f"Samples per advert: {NUM_SAMPLES}")
print(f"Total samples: {len(ADVERTS) * NUM_SAMPLES}")
print("=" * 60)

survey_results = run_survey(
    ADVERTS,
    ADVERT_LABELS,
    NUM_SAMPLES,
    QUESTION,
    REFERENCES,
)

print("\n" + "=" * 60)
print("Survey simulation complete!")
print(f"Total responses collected: {len(survey_results)}")


## 10. Results Analysis


In [ ]:
# Display basic statistics
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

summary = survey_results.groupby("advert_label")["ssr_rating"].agg([
    ("Mean Rating", "mean"),
    ("Std Dev", "std"),
    ("Min", "min"),
    ("Max", "max"),
    ("Count", "count"),
]).round(2)

print(summary)

# Display top-performing advert
best_advert = summary["Mean Rating"].idxmax()
best_rating = summary.loc[best_advert, "Mean Rating"]
print(f"\n🏆 Top performing advert: {best_advert} (Mean SSR: {best_rating:.2f})")


## 11. Visualization - Bar Charts


In [ ]:
# Create bar chart of mean ratings
fig, ax = plt.subplots(figsize=(12, 6))

mean_ratings = survey_results.groupby("advert_label")["ssr_rating"].mean().sort_values(ascending=False)
std_ratings = survey_results.groupby("advert_label")["ssr_rating"].std()

colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(mean_ratings)))
bars = ax.bar(range(len(mean_ratings)), mean_ratings.values, yerr=std_ratings.values,
              color=colors, alpha=0.8, capsize=5)

ax.set_xlabel("Advertisement", fontsize=12, fontweight="bold")
ax.set_ylabel("Mean SSR Rating (1-5 Likert Scale)", fontsize=12, fontweight="bold")
ax.set_title(f"Advertisement Performance: {INTENT_QUESTION.title()}", fontsize=14, fontweight="bold")
ax.set_xticks(range(len(mean_ratings)))
ax.set_xticklabels(mean_ratings.index, rotation=45, ha="right")
ax.set_ylim([1, 5])
ax.grid(axis="y", alpha=0.3)

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, mean_ratings.values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height + std_ratings.iloc[i] + 0.05,
           f'{value:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


## 12. Distribution Analysis


In [ ]:
# Create distribution histograms for each advert
num_adverts = len(survey_results["advert_label"].unique())
fig, axes = plt.subplots(1, num_adverts, figsize=(6 * num_adverts, 5))

if num_adverts == 1:
    axes = [axes]

colors_dist = plt.cm.viridis(np.linspace(0.3, 0.9, num_adverts))

for idx, (advert_label, group) in enumerate(survey_results.groupby("advert_label")):
    ax = axes[idx]
    
    # Create histogram
    ratings = group["ssr_rating"].values
    ax.hist(ratings, bins=np.arange(1, 6.5, 0.5), alpha=0.7, color=colors_dist[idx], edgecolor="black")
    
    ax.set_xlabel("SSR Rating", fontsize=11, fontweight="bold")
    ax.set_ylabel("Frequency", fontsize=11, fontweight="bold")
    ax.set_title(f"{advert_label}\nMean: {ratings.mean():.2f} | Std: {ratings.std():.2f}",
                fontsize=12, fontweight="bold")
    ax.set_xlim([0.5, 5.5])
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## 13. Demographic Analysis


In [ ]:
# Analyze ratings by demographic segments
demographic_keys = list(DEMOGRAPHICS.keys())

for demo_key in demographic_keys:
    print(f"\n{'=' * 60}")
    print(f"Analysis by {demo_key.upper()}")
    print(f"{'=' * 60}")
    
    demo_analysis = survey_results.groupby(["advert_label", demo_key])["ssr_rating"].mean().unstack()
    print(demo_analysis.round(2))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(14, 6))
    demo_analysis.T.plot(kind="bar", ax=ax, alpha=0.8)
    ax.set_xlabel(demo_key.replace("_", " ").title(), fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean SSR Rating", fontsize=12, fontweight="bold")
    ax.set_title(f"Advertisement Performance by {demo_key.replace('_', ' ').title()}",
                fontsize=14, fontweight="bold")
    ax.legend(title="Advertisement", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.set_ylim([1, 5])
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 14. Qualitative Analysis - Sample Responses


In [ ]:
# Display sample qualitative responses
print("\n" + "=" * 60)
print("SAMPLE QUALITATIVE RESPONSES")
print("=" * 60)

for advert_label in survey_results["advert_label"].unique():
    print(f"\n{'*' * 60}")
    print(f"Advertisement: {advert_label}")
    print(f"{'*' * 60}\n")
    
    advert_data = survey_results[survey_results["advert_label"] == advert_label]
    
    # Show samples from different rating levels
    for rating_level in [5, 3, 1]:
        # Find closest rating to this level
        closest_samples = advert_data.iloc[
            (advert_data["ssr_rating"] - rating_level).abs().nsmallest(2).index
        ]
        
        if len(closest_samples) > 0:
            sample = closest_samples.iloc[0]
            print(f"Rating ~{rating_level} (actual: {sample['ssr_rating']:.2f})")
            print(f"Demographics: {sample['age_group']}, {sample['gender']}, {sample['income_level']} income")
            print(f"Response: {sample['response_text']}")
            print()


## 15. Export Results


In [ ]:
# Save results to CSV
output_file = "market_research_results.csv"
survey_results.to_csv(output_file, index=False)
print(f"\n✅ Results saved to: {output_file}")

# Save summary statistics
summary_file = "market_research_summary.csv"
summary.to_csv(summary_file)
print(f"✅ Summary statistics saved to: {summary_file}")
